# Robustness checks

A few things peers flagged about the main models: no baseline/CIs in the results table, `Phase`
maybe letting the model shortcut around heart rate, and the per-individual z-scoring possibly
leaking the held-out person's own data. This checks all three against the HR-only models in
`logistic_regression_training.ipynb` / `ANN_training.ipynb`.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

df = pd.read_csv("HR_data.csv", index_col=0)
df["Frustrated_binary"] = (df["Frustrated"] >= 3).astype(int)
raw_feature_cols = ["HR_Mean", "HR_Median", "HR_std", "HR_Max", "HR_AUC"]

fold_individual_full = df["Individual"].to_numpy()
individuals = np.unique(fold_individual_full)
idx_by_ind = {ind: np.where(fold_individual_full == ind)[0] for ind in individuals}


def cluster_bootstrap_ci(y_true, y_pred_or_proba, metric_fn, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    stats = []
    for _ in range(n_boot):
        sample_inds = rng.choice(individuals, size=len(individuals), replace=True)
        idxs = np.concatenate([idx_by_ind[i] for i in sample_inds])
        stats.append(metric_fn(y_true[idxs], y_pred_or_proba[idxs]))
    stats = np.array(stats)
    return np.percentile(stats, 2.5), np.percentile(stats, 97.5)

## 1. Why we dropped Phase and Round

We also tried training with `Phase` and `Round` added back in (saved as
`results_logreg_with_phase.npz` / `results_ann_with_phase.npz`). `Phase` turned out to be by far
the strongest predictor for both models (logistic regression coefficient $+2.43$, ANN permutation
importance $0.242$, almost double `HR_Mean`'s). That's suspicious: the frustration threshold was
itself picked using phase-level averages (rest $1.21$, task $3.77$, recovery $1.89$), so `Phase`
is basically encoding the label already. Keeping it would mean the model is mostly recognizing the
experiment phase, not learning from heart rate.

In [2]:
with_phase = {
    "Logistic Regression (with Phase/Round)": dict(np.load("results_logreg_with_phase.npz", allow_pickle=True)),
    "ANN (with Phase/Round)": dict(np.load("results_ann_with_phase.npz", allow_pickle=True)),
}

print(f"{'Model':40s} {'Accuracy':>10s} {'Balanced Acc':>14s} {'ROC-AUC':>10s}")
for name, r in with_phase.items():
    acc = accuracy_score(r["oof_true"], r["oof_pred"])
    bal = balanced_accuracy_score(r["oof_true"], r["oof_pred"])
    auc = roc_auc_score(r["oof_true"], r["oof_proba"])
    print(f"{name:40s} {acc:10.3f} {bal:14.3f} {auc:10.3f}")
print("\nCompare to the HR-only primary models below: both metrics drop substantially once")
print("Phase/Round are removed, confirming the shortcut was doing most of the work.")

Model                                      Accuracy   Balanced Acc    ROC-AUC
Logistic Regression (with Phase/Round)        0.702          0.683      0.727
ANN (with Phase/Round)                        0.756          0.722      0.718

Compare to the HR-only primary models below: both metrics drop substantially once
Phase/Round are removed, confirming the shortcut was doing most of the work.


## 2. Majority-class baseline

Just predicts "not frustrated" every time, ignoring all inputs. Balanced accuracy is 0.5 by
definition -- the floor the real models should be compared against.

In [3]:
majority_class = df["Frustrated_binary"].mode()[0]
baseline_pred = np.full(len(df), majority_class)
baseline_acc = accuracy_score(df["Frustrated_binary"], baseline_pred)
baseline_bal = balanced_accuracy_score(df["Frustrated_binary"], baseline_pred)
print(f"Majority-class baseline: accuracy={baseline_acc:.3f}, balanced_accuracy={baseline_bal:.3f}")

Majority-class baseline: accuracy=0.625, balanced_accuracy=0.500


## 3. Bootstrap confidence intervals

95% CI on balanced accuracy for each model, resampling individuals 2000 times so a person never
ends up split across train/test within a resample.

In [4]:
primary = {
    "Logistic Regression": dict(np.load("results_logreg.npz", allow_pickle=True)),
    "ANN": dict(np.load("results_ann.npz", allow_pickle=True)),
}

print(f"{'Model':24s} {'Accuracy':>10s} {'Balanced Acc [95% CI]':>28s} {'ROC-AUC':>10s}")
for name, r in primary.items():
    acc = accuracy_score(r["oof_true"], r["oof_pred"])
    bal = balanced_accuracy_score(r["oof_true"], r["oof_pred"])
    auc = roc_auc_score(r["oof_true"], r["oof_proba"])
    bal_ci = cluster_bootstrap_ci(r["oof_true"], r["oof_pred"], balanced_accuracy_score)
    print(f"{name:24s} {acc:10.3f} {bal:6.3f} [{bal_ci[0]:.3f}, {bal_ci[1]:.3f}] {auc:10.3f}")

Model                      Accuracy        Balanced Acc [95% CI]    ROC-AUC


Logistic Regression           0.595  0.597 [0.516, 0.684]      0.607


ANN                           0.506  0.506 [0.446, 0.581]      0.448


## 4. Comparing the two models more carefully

`model_comparison.ipynb` already has the Wilcoxon test. Here we add a bootstrap CI on the
difference, plus McNemar's test as a cross-check -- though McNemar assumes independent rows, which
isn't true here since each person contributes 12.

In [5]:
lr = primary["Logistic Regression"]
ann = primary["ANN"]

rng = np.random.default_rng(seed=42)
diffs = []
for _ in range(2000):
    sample_inds = rng.choice(individuals, size=len(individuals), replace=True)
    idxs = np.concatenate([idx_by_ind[i] for i in sample_inds])
    bal_lr = balanced_accuracy_score(lr["oof_true"][idxs], lr["oof_pred"][idxs])
    bal_ann = balanced_accuracy_score(ann["oof_true"][idxs], ann["oof_pred"][idxs])
    diffs.append(bal_lr - bal_ann)
diffs = np.array(diffs)

diff_point = balanced_accuracy_score(lr["oof_true"], lr["oof_pred"]) - balanced_accuracy_score(ann["oof_true"], ann["oof_pred"])
diff_ci = np.percentile(diffs, [2.5, 97.5])
print(f"Difference in balanced accuracy (LogReg - ANN): {diff_point:.3f}  95% CI [{diff_ci[0]:.3f}, {diff_ci[1]:.3f}]")

lr_correct = (lr["oof_true"] == lr["oof_pred"])
ann_correct = (ann["oof_true"] == ann["oof_pred"])
lr_only = int(np.sum(lr_correct & ~ann_correct))
ann_only = int(np.sum(~lr_correct & ann_correct))
n_discordant = lr_only + ann_only
mcnemar_p = binomtest(min(lr_only, ann_only), n_discordant, 0.5).pvalue
print(f"\nMcNemar's test: LogReg-only correct={lr_only}, ANN-only correct={ann_only}, n_discordant={n_discordant}")
print(f"McNemar exact p-value: {mcnemar_p:.3f}  (assumes independent rows -- violated here, read as a cross-check only)")

Difference in balanced accuracy (LogReg - ANN): 0.090  95% CI [-0.012, 0.198]

McNemar's test: LogReg-only correct=32, ANN-only correct=17, n_discordant=49
McNemar exact p-value: 0.044  (assumes independent rows -- violated here, read as a cross-check only)


## 5. Does the z-scoring leak?

Each person's z-score uses their own mean/std from all 12 rows, including whichever row is held
out -- a genuinely new person wouldn't have that. We try a stricter version using only their
resting-phase rows to compute the mean/std, and check if it changes anything.

In [6]:
from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve

calib_stats = df[df["Phase"] == "phase1"].groupby("Individual")[raw_feature_cols].agg(["mean", "std"])
df_calib = df.copy()
for col in raw_feature_cols:
    m = df_calib["Individual"].map(calib_stats[(col, "mean")])
    s = df_calib["Individual"].map(calib_stats[(col, "std")])
    df_calib[col] = (df_calib[col] - m) / s

X_calib = df_calib[raw_feature_cols]
y_calib = df_calib["Frustrated_binary"]
groups_calib = df_calib["Individual"]

pipeline = Pipeline([
    ("preprocessor", ColumnTransformer([("hr", StandardScaler(), raw_feature_cols)])),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])
param_grid = {"clf__C": [0.001, 0.01, 0.1, 1, 10, 100], "clf__class_weight": [None, "balanced"]}
outer_cv = LeaveOneGroupOut()
oof_true = np.zeros(len(y_calib), dtype=int)
oof_pred = np.zeros(len(y_calib), dtype=int)
oof_proba = np.zeros(len(y_calib), dtype=float)

for train_idx, test_idx in outer_cv.split(X_calib, y_calib, groups=groups_calib):
    X_tr, X_te = X_calib.iloc[train_idx], X_calib.iloc[test_idx]
    y_tr, y_te = y_calib.iloc[train_idx], y_calib.iloc[test_idx]
    groups_tr = groups_calib.iloc[train_idx]

    inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    search = GridSearchCV(pipeline, param_grid, cv=inner_cv, scoring="balanced_accuracy", n_jobs=-1)
    search.fit(X_tr, y_tr, groups=groups_tr)
    best_model = search.best_estimator_

    inner_oof_proba = cross_val_predict(best_model, X_tr, y_tr, groups=groups_tr, cv=inner_cv, method="predict_proba")[:, 1]
    fpr_in, tpr_in, thr_in = roc_curve(y_tr, inner_oof_proba)
    thr = thr_in[np.argmax(tpr_in - fpr_in)]

    proba_te = best_model.predict_proba(X_te)[:, 1]
    pred_te = (proba_te >= thr).astype(int)
    oof_true[test_idx] = y_te.values
    oof_proba[test_idx] = proba_te
    oof_pred[test_idx] = pred_te

print("HR-only feature set, calibration-window (phase1-only) z-score:")
print(f"  accuracy={accuracy_score(oof_true, oof_pred):.3f}, "
      f"balanced_accuracy={balanced_accuracy_score(oof_true, oof_pred):.3f}, "
      f"ROC-AUC={roc_auc_score(oof_true, oof_proba):.3f}")
lr_acc = accuracy_score(lr['oof_true'], lr['oof_pred'])
lr_bal = balanced_accuracy_score(lr['oof_true'], lr['oof_pred'])
lr_auc = roc_auc_score(lr['oof_true'], lr['oof_proba'])
print(f"(compare to in-sample z-score: accuracy={lr_acc:.3f}, balanced_accuracy={lr_bal:.3f}, ROC-AUC={lr_auc:.3f})")

HR-only feature set, calibration-window (phase1-only) z-score:
  accuracy=0.589, balanced_accuracy=0.576, ROC-AUC=0.577
(compare to in-sample z-score: accuracy=0.595, balanced_accuracy=0.597, ROC-AUC=0.607)


## Summary

Reproduces Table 1 of `task2_report.tex`.

In [7]:
summary_rows = [{"Model": "Baseline (majority class)", "Accuracy": baseline_acc, "Balanced Accuracy": baseline_bal, "ROC-AUC": 0.500}]
for name, r in primary.items():
    summary_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(r["oof_true"], r["oof_pred"]),
        "Balanced Accuracy": balanced_accuracy_score(r["oof_true"], r["oof_pred"]),
        "ROC-AUC": roc_auc_score(r["oof_true"], r["oof_proba"]),
    })
for name, r in with_phase.items():
    summary_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(r["oof_true"], r["oof_pred"]),
        "Balanced Accuracy": balanced_accuracy_score(r["oof_true"], r["oof_pred"]),
        "ROC-AUC": roc_auc_score(r["oof_true"], r["oof_proba"]),
    })
pd.DataFrame(summary_rows).round(3)

,Model,Accuracy,Balanced Accuracy,ROC-AUC
0,Baseline (majority class),0.625,0.500,0.500
1,Logistic Regression,0.595,0.597,0.607
2,ANN,0.506,0.506,0.448
3,Logistic Regression (with Phase/Round),0.702,0.683,0.727
4,ANN (with Phase/Round),0.756,0.722,0.718
